# Bài 24 · Mixture of Experts (MoE): mô hình khổng lồ, nhưng mỗi token chỉ đánh thức vài chuyên gia

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/24-mixture-of-experts.ipynb)

Notebook đi kèm [Bài 24 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/mixture-of-experts/) (demo tương tác nằm ở trang bài học).
Ta tự viết bằng numpy một lớp MoE (bộ định tuyến, top-k, các chuyên gia nằm trong một khối có đường dư), huấn luyện nó
trên MNIST thật, và đếm lại tham số của Mixtral 8x7B và DeepSeek-V3 từ config.json. Hai câu hỏi:
**vì sao một mô hình có 46,7 tỉ tham số mà mỗi token chỉ dùng 12,9 tỉ**, và
**chuyện gì xảy ra khi không ai ép bộ định tuyến chia đều việc?**

**Bạn sẽ làm:**
1. Viết một lớp MoE: điểm softmax của bộ định tuyến, chọn top-k, chuẩn hóa cổng, cộng đầu ra các chuyên gia vào đường
   dư; viết tay lan truyền ngược và kiểm tra bằng sai phân hữu hạn.
2. Đếm tham số tổng và tham số hoạt động của Mixtral 8x7B và DeepSeek-V3 từ config.json, khớp tới từng tham số với
   checkpoint thật.
3. Huấn luyện MoE trên MNIST, nhìn bộ định tuyến sụp khi không cân bằng, rồi sửa bằng mất mát phụ của Switch
   Transformer và độ lệch chỉ dùng để chọn của DeepSeek-V3.
4. Giới hạn dung lượng của mỗi chuyên gia và đếm token bị bỏ.
5. Chạy mô hình int8 của demo trên 10.000 ảnh kiểm tra, xem chuyên gia nào nhận chữ số nào, và so MoE với mạng dày.

**Cần biết trước:** [Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/) (lớp FFN
và luồng dư) và [Bài 23 · Quy luật mở rộng](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/quy-luat-mo-rong/) (tham số
và phép tính); lan truyền ngược của [Bài 06](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/lan-truyen-nguoc/), Adam
của [Bài 07](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/bo-toi-uu/) và đường dư của
[Bài 13](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/resnet/). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải
MNIST (~11 MB) và ba tệp kết quả nhỏ của demo lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra, và trùng với số trong demo. Riêng các lần huấn luyện đầy
đủ (33 cấu hình × 3 seed × 10 epoch, vài phút trên 16 nhân) được nạp từ đúng tệp mà demo dùng; notebook chạy lại một
phiên bản ngắn của từng cơ chế và kiểm tra những gì tính lại được.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi.

In [ ]:
import base64
import gzip
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng điểm, nét gạch, nhãn):
# không cân bằng = cam, ô vuông; có cân bằng = xanh, chấm tròn; mạng dày = xanh lá, hình thoi
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
SEQ = LinearSegmentedColormap.from_list("trang_xanh", ["#f7f9fc", BLUE, "#0b3a75"])  # 0 → lớn
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


def span(values, digits=2, scale=1.0):
    """Khoảng min–max qua các seed, như demo: 97,40–97,57 (một số nếu hai đầu bằng nhau khi làm tròn)."""
    lo, hi = scale * min(values), scale * max(values)
    return vn(lo, digits) if round(lo, digits) == round(hi, digits) else f"{vn(lo, digits)}–{vn(hi, digits)}"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """Số rất nhỏ hoặc rất lớn: 4,1·10¹⁴."""
    e = math.floor(math.log10(abs(x)))
    return f"{vn(x / 10**e, digits)}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(vn(13007, 0), vn(0.1, 1), vn(-4.3805, 4), pct(0.993527), span([0.974, 0.9757], scale=100), sci(4.1e14))

## 2 · Dữ liệu

MNIST (LeCun và cộng sự, 1998): 60.000 ảnh huấn luyện và 10.000 ảnh kiểm tra, chữ số viết tay xám 28×28. Ô dưới tải
4 tệp gốc và kiểm tra mã MD5, nên ai chạy cũng có đúng cùng một dữ liệu. Giống các lần chạy của demo, ta chia:
- **huấn luyện:** 55.000 ảnh đầu (mọi mạng học trên phần này);
- **kiểm định:** 5.000 ảnh cuối của tệp huấn luyện (demo chọn tốc độ học $\eta$ ở đây);
- **kiểm tra:** 10.000 ảnh của tệp kiểm tra, chỉ để báo kết quả.

Ô cũng nạp ba tệp kết quả mà demo dùng (`fetch_json`, có kiểm tra MD5):
- `moe-counts.json`: số tham số của 10 checkpoint thật, đếm từ header của từng tệp safetensors, theo từng phần;
- `moe-mnist.json`: 33 cấu hình MoE và mạng dày trên MNIST × 3 seed × 10 epoch (tải chuyên gia, chuyên gia chết,
  token bị bỏ, ma trận chữ số × chuyên gia…);
- `moe-weights.json`: mô hình MoE mà demo chạy trong trình duyệt (seed 0, trọng số int8), kèm 64 ảnh kiểm tra.

In [ ]:
URL = "https://ossci-datasets.s3.amazonaws.com/mnist/"
FILES = {  # mã MD5 chuẩn của MNIST
    "train-images-idx3-ubyte.gz": "f68b3c2dcbeaaa9fbdd348bbdeb94873",
    "train-labels-idx1-ubyte.gz": "d53e105ee54ea40749a09fcbcd1e9432",
    "t10k-images-idx3-ubyte.gz": "9fb629c4189551a2d022fa330f9573f3",
    "t10k-labels-idx1-ubyte.gz": "ec29112dd5afa0611ce80d1b7f02629c",
}
DATA = Path("mnist")


def fetch():
    DATA.mkdir(exist_ok=True)
    for name, md5 in FILES.items():
        p = DATA / name
        if not p.exists():
            urllib.request.urlretrieve(URL + name, p)
        assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"


def load(kind):
    """Định dạng IDX: 16 byte đầu của tệp ảnh (8 byte của tệp nhãn) là phần mô tả, còn lại là các byte 0–255."""
    with gzip.open(DATA / f"{kind}-images-idx3-ubyte.gz") as f:
        images = np.frombuffer(f.read(), np.uint8, offset=16).reshape(-1, 784)
    with gzip.open(DATA / f"{kind}-labels-idx1-ubyte.gz") as f:
        labels = np.frombuffer(f.read(), np.uint8, offset=8)
    return images, labels


# Kết quả của các lần huấn luyện dài (cùng tệp với demo trên web). Khi xuất bản, build.py thay chuỗi giữ chỗ trong
# urlretrieve dưới đây bằng thư mục data/ của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


fetch()
train_img, train_lab = load("train")
test_img, test_lab = load("t10k")
counts = fetch_json("moe-counts.json", "997b5403cce3c91f064a79fc2cacc3e4")["models"]
sweep = fetch_json("moe-mnist.json", "bf417d9bf0dfa1ec75f8a7d9f8e6e36c")
shipped = fetch_json("moe-weights.json", "891715ae39ce1d0e1aed71fa948b9786")
size = sum((Path("data") / f).stat().st_size for f in ["moe-counts.json", "moe-mnist.json", "moe-weights.json"])
print(
    f"huấn luyện: {vn(len(train_img), 0)} ảnh · kiểm tra: {vn(len(test_img), 0)} ảnh · 3 tệp của demo: {vn(size / 1e6, 2)} MB"
)

Điểm ảnh được chia cho 255 và lưu ở float32, như các lần chạy của demo. Hình dưới là 12 ảnh kiểm tra đầu tiên; chúng
cũng là 12 ảnh đầu trong 64 ảnh mà demo cho bạn chọn (mục 8).

In [ ]:
Xtr, ytr = (train_img[:55000] / 255.0).astype(np.float32), train_lab[:55000].astype(np.int64)
Xte, yte = (test_img / 255.0).astype(np.float32), test_lab.astype(np.int64)
BATCH = 128
print(f"huấn luyện {vn(len(Xtr), 0)} ảnh ({len(Xtr) // BATCH} lô 128 ảnh mỗi epoch) · kiểm tra {vn(len(Xte), 0)} ảnh")

fig, axes = plt.subplots(1, 12, figsize=(10, 1.2))
for ax, img, lab in zip(axes, Xte, yte):
    ax.imshow(img.reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    ax.set_title(str(lab), fontsize=10)
    ax.axis("off")
plt.show()

## 3 · Một lớp MoE từ đầu

Trong mỗi khối Transformer (Bài 17), sau attention là một lớp FFN: một MLP nhỏ chạy riêng cho từng token, và phần lớn
tham số của mô hình nằm ở đây. MoE thay lớp FFN đó bằng $N$ lớp FFN, gọi là **chuyên gia**, cùng một
**bộ định tuyến** (router). Với vectơ hàng $h$ của một token:

$$s = \operatorname{softmax}(h W_r), \qquad \mathcal T = \operatorname{TopK}(s + b,\; k), \qquad
g_i = \frac{s_i}{\sum_{j \in \mathcal T} s_j}, \qquad u = h + \sum_{i \in \mathcal T} g_i\, \mathrm{FFN}_i(h)$$

- $W_r$ có $d \times N$ phần tử: bộ định tuyến chỉ là **một lớp tuyến tính** chấm điểm $N$ chuyên gia.
- $\mathcal T$ là $k$ chuyên gia điểm cao nhất. $b$ là độ lệch cân bằng tải của DeepSeek-V3 (mục 6): nó chỉ dùng để
  *chọn*, không đi vào $g_i$; tạm thời $b = 0$. Nó không phải độ lệch $b$ trong $xW + b$ của Bài 01.
- $g_i$ là **trọng số cổng**, chuẩn hóa trên $k$ chuyên gia được chọn (Mixtral). Switch Transformer ($k = 1$) giữ
  nguyên $g_i = s_i$; nếu chuẩn hóa thì với $k = 1$ cổng luôn bằng 1 và bộ định tuyến không học được gì từ bài toán.
- Chuyên gia không được chọn **không chạy** cho token đó. $u$ cộng đầu ra các chuyên gia vào **đường dư** $h$, như mọi
  lớp FFN trong Transformer (và Bài 13).

Mạng của ta là mạng của demo (`reel/moe/mnist.py`): ảnh 784 điểm → thân $h = \mathrm{ReLU}(xW_0 + b_0)$ với 64 chiều →
lớp MoE với 8 chuyên gia 64 → 32 → 64, $\mathrm{FFN}_i(v) = \mathrm{ReLU}(vA_i + a_i)B_i + c_i$ → lớp đầu ra 10 logit.
Ô dưới khởi tạo mạng đúng như demo và viết bộ định tuyến.

In [ ]:
def init_moe(seed, E=8, H=32, d=64, d_in=784, n_cls=10, dtype=np.float32):
    """Khởi tạo như reel/moe/mnist.py: He cho lớp đứng trước ReLU, B của chuyên gia ở nửa thang, W_r ~ N(0; 0,02²)."""
    rng = np.random.default_rng(seed)

    def he(fan_in, shape):
        return (rng.standard_normal(shape) * math.sqrt(2.0 / fan_in)).astype(dtype)

    P = {"W0": he(d_in, (d_in, d)), "b0": np.zeros(d, dtype)}
    P["A"], P["a"] = he(d, (E, d, H)), np.zeros((E, H), dtype)
    P["B"], P["c"] = (rng.standard_normal((E, H, d)) * math.sqrt(1.0 / H) * 0.5).astype(dtype), np.zeros((E, d), dtype)
    P["Wo"] = (rng.standard_normal((d, n_cls)) * math.sqrt(1.0 / d)).astype(dtype)
    P["bo"] = np.zeros(n_cls, dtype)
    P["Wr"] = (rng.standard_normal((d, E)) * 0.02).astype(dtype)  # bộ định tuyến: không có độ lệch, như Mixtral
    return P


def softmax(l):
    e = np.exp(l - l.max(1, keepdims=True))
    return e / e.sum(1, keepdims=True)


def route(h, Wr, k, bias=None, gate="renorm"):
    """Điểm s = softmax(h W_r); chọn k chuyên gia theo s + b (b chỉ để chọn); cổng g chuẩn hóa ("renorm") hoặc thô."""
    s = softmax(h @ Wr)
    key = s if bias is None else s + bias
    sel = np.argsort(-key, axis=1, kind="stable")[:, :k]  # k chuyên gia, tốt nhất trước; hòa thì chỉ số nhỏ hơn
    sv = np.take_along_axis(s, sel, 1)
    g = sv / sv.sum(1, keepdims=True) if gate == "renorm" else sv
    return s, sel, g


def ffn(v, P, e):
    """Chuyên gia e: một MLP nhỏ d → H → d."""
    return np.maximum(v @ P["A"][e] + P["a"][e], 0) @ P["B"][e] + P["c"][e]


P0 = init_moe(seed=0)
n_par = {key: v.size for key, v in P0.items()}
print("số phần tử:", {key: vn(n, 0) for key, n in n_par.items()})
expert = n_par["A"] // 8 + n_par["a"] // 8 + n_par["B"] // 8 + n_par["c"] // 8
trunk = n_par["W0"] + n_par["b0"] + n_par["Wo"] + n_par["bo"]
print(f"một chuyên gia: {vn(expert, 0)} tham số · thân + đầu ra: {vn(trunk, 0)} · bộ định tuyến: {vn(n_par['Wr'], 0)}")

Một chuyên gia có $64 \cdot 32 + 32 + 32 \cdot 64 + 64$ tham số. Bộ định tuyến rất nhỏ: một ma trận $64 \times 8$.

Thử đi từng bước với một token, ở đây là ảnh kiểm tra đầu tiên (số 7), trên mạng **chưa học**. Ô dưới in điểm $s$ của
8 chuyên gia, hai chuyên gia được chọn và cổng của chúng, rồi cộng đầu ra của hai chuyên gia vào đường dư bằng một
vòng lặp.

In [ ]:
k = 2
x = Xte[:1]  # một hàng: một token
h = np.maximum(x @ P0["W0"] + P0["b0"], 0)
s, sel, g = route(h, P0["Wr"], k)
print("s =", " ".join(vn(v, 3) for v in s[0]), f" (tổng {vn(s.sum(), 3)})")
print(f"top-{k}: chuyên gia {sel[0].tolist()}, cổng g = {' và '.join(vn(v, 3) for v in g[0])} (tổng {vn(g.sum(), 3)})")
u = h.copy()  # đường dư
for i, e in enumerate(sel[0]):
    u += g[0, i] * ffn(h, P0, e)  # chỉ 2 trong 8 chuyên gia chạy
print(f"‖h‖ = {vn(np.linalg.norm(h), 3)}, ‖u − h‖ = {vn(np.linalg.norm(u - h), 3)}: phần MoE cộng vào đường dư")

Lúc khởi tạo, $W_r$ rất nhỏ nên $hW_r \approx 0$ và mọi $s_i$ đều gần $1/8 = 0{,}125$: bộ định tuyến chưa có ý kiến
gì, và hai chuyên gia được chọn gần như tình cờ. Mục 5 cho thấy chính điều này dẫn tới sụp đổ.

Tính từng token một thì rất chậm. Phần cứng thật làm ngược lại: **gom các token đã chọn cùng một chuyên gia** (dispatch),
chạy chuyên gia đó một lần trên cả nhóm, rồi trả kết quả về chỗ cũ (combine). Hàm `forward` dưới làm đúng như vậy, có
thêm hai thứ dùng ở mục 7 và 8: giới hạn dung lượng `cf` (tạm bỏ qua) và `off` để tắt hẳn một chuyên gia.
Ô cuối so cách gom theo chuyên gia với vòng lặp từng token trên 12 ảnh.

In [ ]:
def capacity_keep(sel, E, cf):
    """Suất nào vừa dung lượng C = ⌈cf·n·k/N⌉ của chuyên gia (mục 7). cf = None: không giới hạn."""
    n, k = sel.shape
    if cf is None:
        return np.ones_like(sel, bool)
    C = math.ceil(cf * n * k / E)
    flat = sel.T.ravel()  # suất j·n + t = lựa chọn thứ j của token t: mọi lựa chọn thứ nhất xếp hàng trước
    onehot = np.zeros((n * k, E), np.int64)
    onehot[np.arange(n * k), flat] = 1
    rank = np.cumsum(onehot, 0)[np.arange(n * k), flat]  # vị trí (từ 1) trong hàng đợi của chuyên gia đó
    return (rank <= C).reshape(k, n).T


def forward(P, X, k, bias=None, gate="renorm", cf=None, off=None):
    """z = (h + Σ_{i∈𝒯} gᵢ FFNᵢ(h)) Wo + bo với h = ReLU(x W0 + b0). off[e] = True tắt chuyên gia e."""
    E = P["A"].shape[0]
    hp = X @ P["W0"] + P["b0"]
    h = np.maximum(hp, 0)
    s, sel, g = route(h, P["Wr"], k, bias, gate)
    kept = capacity_keep(sel, E, cf)
    if off is not None:
        kept &= ~off[sel]  # suất của chuyên gia bị tắt bị bỏ, như suất tràn dung lượng
    u = h.copy()  # đường dư
    cache = []
    for e in range(E):  # gom: mọi token đã chọn chuyên gia e (và còn chỗ)
        rows, slot = np.nonzero((sel == e) & kept)
        q = h[rows] @ P["A"][e] + P["a"][e]
        r = np.maximum(q, 0)
        o = r @ P["B"][e] + P["c"][e]
        u[rows] += g[rows, slot][:, None] * o  # trả về: mỗi token chọn một chuyên gia nhiều nhất một lần
        cache.append((rows, slot, q, r, o))
    z = u @ P["Wo"] + P["bo"]
    return z, {"hp": hp, "h": h, "s": s, "sel": sel, "g": g, "kept": kept, "u": u, "cache": cache}


X12 = Xte[:12]
z12, C12 = forward(P0, X12, k=2)
u_loop = C12["h"].copy()
for t in range(12):  # vòng lặp từng token, như ô trên
    for i, e in enumerate(C12["sel"][t]):
        u_loop[t] += C12["g"][t, i] * ffn(C12["h"][t : t + 1], P0, e)[0]
assert np.allclose(u_loop, C12["u"], atol=1e-5)
print("hai cách tính cho cùng u; chuyên gia được chọn cho 12 ảnh:", C12["sel"].tolist())
load12 = np.bincount(C12["sel"].ravel(), minlength=8)
print("số suất mỗi chuyên gia nhận:", load12.tolist(), f"(tổng {load12.sum()} = 12 token × k = 2)")

Mỗi token đi qua một cặp chuyên gia riêng, và mỗi chuyên gia nhận một số token khác nhau: đó là **tải** (load) của nó.
Mỗi token chiếm $k$ **suất tính** (slot), nên tổng tải bằng $12 \cdot 2$.

Một chi tiết của Mixtral: bài báo viết cổng là $\operatorname{softmax}(\operatorname{TopK}(hW_r))$, tức softmax chỉ
trên $k$ logit lớn nhất. Cách ta viết (softmax trên cả $N$, rồi chia cho tổng của $k$ điểm được chọn) là cùng một thứ,
vì mẫu số $\sum_j e^{l_j}$ của softmax đầy đủ triệt tiêu:

In [ ]:
l12 = C12["h"] @ P0["Wr"]
lt = np.take_along_axis(l12, C12["sel"], 1)  # k logit lớn nhất của mỗi token
gate_mixtral = softmax(lt)
print(f"|cổng của ta − softmax(TopK(logit))| lớn nhất: {sci(np.abs(C12['g'] - gate_mixtral).max())} (float32)")
assert np.allclose(C12["g"], gate_mixtral, atol=1e-6)

### Lan truyền ngược qua bộ định tuyến

Việc *chọn* $\mathcal T$ không khả vi: đổi $W_r$ một chút thì tập được chọn hoặc giữ nguyên, hoặc nhảy. Gradient đến
bộ định tuyến chỉ qua **cổng** $g_i$: $\partial \mathcal L/\partial g_i = \delta_u \cdot \mathrm{FFN}_i(h)$, với
$\delta_u = \partial \mathcal L / \partial u$. Với cổng chuẩn hóa $g_i = s_i/S$, $S = \sum_{j\in\mathcal T} s_j$:

$$\frac{\partial \mathcal L}{\partial s_i} = \frac{1}{S}\Big(\frac{\partial \mathcal L}{\partial g_i}
- \sum_{j \in \mathcal T} g_j \frac{\partial \mathcal L}{\partial g_j}\Big) \;\; (i \in \mathcal T), \qquad
\frac{\partial \mathcal L}{\partial s_i} = 0 \;\; (i \notin \mathcal T)$$

rồi qua softmax như Bài 04. Chuyên gia không được chọn không nhận gradient nào, kể cả qua bộ định tuyến: nó chỉ học
khi có token chọn nó. `loss_grad` cũng nhận mất mát phụ của mục 6 (`alpha`); tạm thời `alpha = 0`.

In [ ]:
def loss_grad(P, X, y, k, bias=None, gate="renorm", cf=None, alpha=0.0):
    """Cross-entropy (+ α·mất mát phụ) và gradient; trả về ce, aux, G, tải của từng chuyên gia, số suất bị bỏ."""
    E, n = P["A"].shape[0], len(X)
    z, C = forward(P, X, k, bias, gate, cf)
    p = softmax(z)
    ce = float(-np.log(p[np.arange(n), y] + 1e-30).mean())
    dz = p.copy()
    dz[np.arange(n), y] -= 1
    dz /= n
    G = {"Wo": C["u"].T @ dz, "bo": dz.sum(0)}
    du = dz @ P["Wo"].T
    dh = du.copy()  # đường dư
    for key in ("A", "a", "B", "c"):
        G[key] = np.zeros_like(P[key])
    dg = np.zeros_like(C["g"])
    for e, (rows, slot, q, r, o) in enumerate(C["cache"]):
        if len(rows) == 0:
            continue  # chuyên gia không được chọn: gradient bằng 0
        dd = du[rows]
        dg[rows, slot] = (dd * o).sum(1)  # ∂L/∂gᵢ = δu · FFNᵢ(h)
        do = C["g"][rows, slot][:, None] * dd
        G["B"][e], G["c"][e] = r.T @ do, do.sum(0)
        dq = (do @ P["B"][e].T) * (q > 0)
        G["A"][e], G["a"][e] = C["h"][rows].T @ dq, dq.sum(0)
        dh[rows] += dq @ P["A"][e].T
    s, sel, g = C["s"], C["sel"], C["g"]
    sv = np.take_along_axis(s, sel, 1)
    dsv = (dg - (dg * g).sum(1, keepdims=True)) / sv.sum(1, keepdims=True) if gate == "renorm" else dg
    ds = np.zeros_like(s)
    np.put_along_axis(ds, sel, dsv, 1)
    aux = 0.0
    if alpha > 0:  # mất mát phụ của Switch (mục 6): α·N·Σ fᵢ Pᵢ; f không khả vi, P = trung bình của s
        f = np.bincount(sel.ravel(), minlength=E) / (n * k)
        aux = float(alpha * E * (f * s.mean(0)).sum())
        ds += alpha * E * f[None] / n
    dl = s * (ds - (ds * s).sum(1, keepdims=True))  # qua softmax
    G["Wr"] = C["h"].T @ dl
    dh += dl @ P["Wr"].T
    dhp = dh * (C["hp"] > 0)
    G["W0"], G["b0"] = X.T @ dhp, dhp.sum(0)
    load = np.bincount(sel.ravel(), minlength=E)
    return ce, aux, G, load, int((~C["kept"]).sum())

Kiểm tra bằng sai phân hữu hạn (Bài 06) trên một mạng nhỏ ở float64: 5 đầu vào, 4 chiều, 4 chuyên gia, 7 token, với
cả cổng chuẩn hóa và cổng thô, top-1 và top-2, có mất mát phụ, có giới hạn dung lượng. Mỗi phần tử của mỗi tham số
được đẩy $\pm\varepsilon$, và ô in sai số tương đối lớn nhất giữa hai cách tính.

In [ ]:
def fd_check(P, G, *args, eps=1e-6):
    """Sai số tương đối lớn nhất giữa G và (L(θ + ε) − L(θ − ε)) / 2ε, với L = ce + aux của loss_grad(P, *args)."""

    def total():
        return sum(loss_grad(P, *args)[:2])

    worst = 0.0
    for key in P:
        for i in np.ndindex(P[key].shape):
            old = P[key][i]
            P[key][i] = old + eps
            lp = total()
            P[key][i] = old - eps
            lm = total()
            P[key][i] = old
            num = (lp - lm) / (2 * eps)
            worst = max(worst, abs(num - G[key][i]) / max(1e-4, abs(num) + abs(G[key][i])))
    return worst


rng = np.random.default_rng(0)
cases = [(2, "renorm", 0.3, None), (2, "raw", 0.3, None), (1, "raw", 0.3, None), (2, "renorm", 0.0, 0.6)]
worst_all = 0.0
for k_, gate_, alpha_, cf_ in cases:
    Pt = init_moe(seed=int(rng.integers(1000)), E=4, H=3, d=4, d_in=5, n_cls=3, dtype=np.float64)
    for key in ("a", "c", "b0"):
        Pt[key] = rng.standard_normal(Pt[key].shape) * 0.1
    Pt["Wr"] = rng.standard_normal(Pt["Wr"].shape)
    Xt, yt_ = rng.standard_normal((7, 5)), rng.integers(0, 3, 7)
    b_ = rng.standard_normal(4) * 0.05  # một độ lệch chọn bất kỳ
    args = (Xt, yt_, k_, b_, gate_, cf_, alpha_)
    _, _, Gt, _, dropped = loss_grad(Pt, *args)
    w = fd_check(Pt, Gt, *args)
    worst_all = max(worst_all, w)
    print(
        f"top-{k_}, cổng {gate_:6s}, α = {vn(alpha_, 1)}, cf = {vn(cf_, 1) if cf_ else '∞'}: sai số tương đối lớn nhất {sci(w)}"
        + (f" ({dropped} suất bị bỏ)" if cf_ else "")
    )
assert worst_all < 1e-5
print(f"lớn nhất qua 4 trường hợp: {sci(worst_all)}")

Mọi gradient khớp với sai phân hữu hạn (sai số tương đối dưới $10^{-5}$), kể cả khi có suất bị bỏ vì giới hạn dung
lượng: suất bị bỏ đơn giản là không có trong tổng, nên cũng không nhận gradient.

## 4 · Tham số tổng và tham số hoạt động

Trước khi huấn luyện, hãy đếm. Một mô hình MoE có hai con số: **tổng tham số** (phải nằm trong bộ nhớ) và **tham số
hoạt động** (mỗi token thực sự dùng). Ta tính cả hai cho Mixtral 8x7B chỉ từ config.json của nó, và so với số đếm từ
header của từng tệp safetensors trong checkpoint thật (`reel/moe/counts.py`, lưu trong `moe-counts.json`).

### 🤔 Dự đoán trước
Mixtral 8x7B có 8 chuyên gia mỗi lớp, và tên gọi gợi ý "8 lần một mô hình 7 tỉ". Tổng số tham số là bao nhiêu? Mỗi
token dùng 2 chuyên gia: nó tốn gấp mấy lần khi chỉ dùng 1?

In [ ]:
# config.json của mistralai/Mixtral-8x7B-v0.1 ở commit fc7ac94 (chỉ các trường công thức cần)
MIXTRAL = {
    "hidden_size": 4096,  # d
    "num_hidden_layers": 32,  # L
    "num_attention_heads": 32,
    "num_key_value_heads": 8,  # grouped-query attention: 8 đầu K/V cho 32 đầu Q
    "vocab_size": 32000,
    "intermediate_size": 14336,  # độ rộng lớp ẩn của mỗi chuyên gia (SwiGLU)
    "num_local_experts": 8,  # N
    "num_experts_per_tok": 2,  # k
    "tie_word_embeddings": False,
}
MISTRAL_7B = {key: v for key, v in MIXTRAL.items() if key not in ("num_local_experts", "num_experts_per_tok")}


def llama_style_params(cfg, k=None):
    """Tham số theo từng phần của một mô hình kiểu Llama/Mistral/Mixtral (không có độ lệch), từ config.json."""
    d, L, V = cfg["hidden_size"], cfg["num_hidden_layers"], cfg["vocab_size"]
    h, kv = cfg["num_attention_heads"], cfg["num_key_value_heads"]
    hd = d // h  # chiều của một đầu
    E = cfg.get("num_local_experts", 1)
    k = k or cfg.get("num_experts_per_tok", 1)
    ffn = 3 * d * cfg["intermediate_size"]  # SwiGLU: ba ma trận gate, up, down
    parts = {
        "embedding": V * d,
        "lớp đầu ra": 0 if cfg["tie_word_embeddings"] else V * d,
        "attention": L * (d * h * hd + 2 * d * kv * hd + h * hd * d),  # Q, K, V, O
        "chuẩn hóa": L * 2 * d + d,  # 2 RMSNorm mỗi khối + 1 ở cuối
        "bộ định tuyến": L * d * E if E > 1 else 0,  # W_r: d × N mỗi lớp
        "FFN / chuyên gia": L * E * ffn,
    }
    total = sum(parts.values())
    active = total - L * E * ffn + L * k * ffn  # mọi phần dùng chung + k trong N chuyên gia ở mỗi lớp
    return parts, total, active


parts_mx, total_mx, active_mx = llama_style_params(MIXTRAL)
for name, n in parts_mx.items():
    print(f"{name:>17}: {vn(n, 0):>15} ({pct(n / total_mx, 1)})")
print(f"tổng: {vn(total_mx, 0)} ≈ {vn(total_mx / 1e9, 1)} tỉ · hoạt động (top-2): {vn(active_mx / 1e9, 1)} tỉ")
ref = counts["mixtral-8x7b"]  # đếm từ 995 tensor trong header safetensors của checkpoint
assert (total_mx, active_mx) == (ref["total"], ref["active"]), "khác với checkpoint: hãy kiểm tra lại config"
assert (round(total_mx / 1e9, 1), round(active_mx / 1e9, 1)) == (46.7, 12.9)
print(f"khớp tới từng tham số với {ref['tensors']} tensor của checkpoint")

**46,7 tỉ, không phải 8 × 7 = 56 tỉ**: chỉ các lớp FFN được nhân 8, còn attention, embedding và chuẩn hóa dùng chung.
Các chuyên gia chiếm gần hết mô hình, nhưng mỗi token chỉ chạy 2 trong 8 chuyên gia ở mỗi lớp. Bài báo Mixtral làm
tròn hai con số này thành 47B và 13B.

Còn top-1 so với top-2:

In [ ]:
_, total_dense, _ = llama_style_params(MISTRAL_7B)
_, _, active_k1 = llama_style_params(MIXTRAL, k=1)
print(f"Mixtral với k = 1: {vn(active_k1, 0)} tham số mỗi token")
print(f"Mistral 7B (dày, cùng kiến trúc): {vn(total_dense, 0)}; chênh lệch {vn(active_k1 - total_dense, 0)}")
print(f"top-2 / top-1 = {vn(active_mx / active_k1, 2)} lần")
assert active_k1 - total_dense == parts_mx["bộ định tuyến"]  # đúng bằng các ma trận W_r
assert total_dense == counts["mistral-7b"]["total"]
assert round(active_mx / active_k1, 2) == 1.78

Với $k = 1$, mỗi token của Mixtral dùng đúng bằng Mistral 7B cộng các ma trận định tuyến: hai mô hình cùng kiến trúc,
Mixtral có 8 bản FFN thay vì 1. Top-2 tốn **1,78 lần** top-1 chứ không phải 2 lần, vì phần dùng chung (attention,
embedding) chạy một lần dù $k$ bằng mấy.

Hình dưới vẽ ba thanh, chiều dài là số tham số: Mistral 7B, phần Mixtral chạy cho mỗi token, và toàn bộ Mixtral.
Phần xám là phần dùng chung; mỗi ô xanh có viền là một chuyên gia (tính qua cả 32 lớp), ô xanh lá gạch chéo là FFN
của mạng dày.

In [ ]:
shared_mx = total_mx - parts_mx["FFN / chuyên gia"]
per_expert = parts_mx["FFN / chuyên gia"] // 8
fig, ax = plt.subplots(figsize=(9, 2.6))
rows = [("Mistral 7B (dày)", 0, 1), ("Mixtral, mỗi token (k = 2)", 2, 0), ("Mixtral, tổng (trong bộ nhớ)", 8, 0)]
for y, (label, n_exp, dense) in enumerate(rows):
    ax.barh(y, shared_mx / 1e9, color=GREY, edgecolor="white", height=0.6)
    for j in range(n_exp):
        ax.barh(y, per_expert / 1e9, left=(shared_mx + j * per_expert) / 1e9, color=BLUE, edgecolor="white", height=0.6)
    if dense:
        ax.barh(y, per_expert / 1e9, left=shared_mx / 1e9, color=GREEN, hatch="//", edgecolor="white", height=0.6)
    end = shared_mx + max(n_exp, dense) * per_expert
    ax.text(end / 1e9 + 0.5, y, f"{vn(end / 1e9, 1)} tỉ", va="center", fontsize=10)
ax.set_yticks(range(3), [r[0] for r in rows])
ax.invert_yaxis()  # đọc từ trên xuống: dày → mỗi token → tổng
ax.set(xlabel="tham số (tỉ)", xlim=(0, 53))
ax.set_title(f"Phần dùng chung {vn(shared_mx / 1e9, 2)} tỉ (xám) + mỗi chuyên gia {vn(per_expert / 1e9, 2)} tỉ (xanh)")
vn_axes(fig)
plt.show()

### Tiết kiệm phép tính, không tiết kiệm bộ nhớ

Chuyên gia không được chọn vẫn phải nằm sẵn trong bộ nhớ, vì token kế tiếp có thể chọn nó. Ở bf16 (2 byte mỗi tham
số), trọng số của Mixtral chiếm:

In [ ]:
GiB = 2**30
print(
    f"{vn(total_mx, 0)} × 2 byte = {vn(2 * total_mx / GiB, 1)} GiB, cho {vn(active_mx / 1e9, 1)} tỉ tham số mỗi token"
)
assert 2 * total_mx == counts["mixtral-8x7b"]["bytes"]  # tổng kích thước tensor trong các tệp safetensors (bf16)


def touched(N, k, B):
    """Phần chuyên gia mà một lô B token chạm vào, nếu mỗi token chọn k trong N một cách đều và độc lập."""
    return 1 - (1 - k / N) ** B


for N_, k_, name in [(8, 2, "Mixtral"), (256, 8, "DeepSeek-V3")]:
    print(f"{name}: lô " + ", ".join(f"{B} token chạm {pct(touched(N_, k_, B), 0)}" for B in (1, 8, 64, 512)))

Tệp checkpoint của Mixtral 8x7B nặng **87,0 GiB** (vì $46{,}7 \cdot 10^9 \cdot 2$ byte). Và dù từng token chỉ đọc vài
chuyên gia, một lô vài chục token đã chạm gần hết: với định tuyến đều và độc lập, một lô 8 token chạm 90% số chuyên
gia của Mixtral. (Định tuyến thật không độc lập, nên con số thật thấp hơn; đây chỉ là minh họa.) MoE giảm phép tính
trên mỗi token, không giảm bộ nhớ. Mô hình lớn phải chia chuyên gia ra nhiều GPU và gửi token qua lại giữa chúng.

### DeepSeek-V3: 256 chuyên gia nhỏ và một chuyên gia dùng chung

DeepSeek-V3 khác Mixtral ở ba chỗ: attention kiểu MLA (nén K/V qua một hạng thấp), 3 lớp đầu là FFN dày, và mỗi lớp MoE
có **256 chuyên gia nhỏ** (lớp ẩn 2.048 thay vì 18.432) chọn 8, cộng **1 chuyên gia dùng chung** luôn chạy.
Công thức dài hơn nhưng cùng ý: hoạt động = mọi phần dùng chung + $k/N$ của các chuyên gia định tuyến.

In [ ]:
# config.json của deepseek-ai/DeepSeek-V3 ở commit e815299 (chỉ các trường công thức cần)
DSV3 = {
    "hidden_size": 7168, "num_hidden_layers": 61, "num_attention_heads": 128, "vocab_size": 129280,
    "intermediate_size": 18432, "moe_intermediate_size": 2048, "first_k_dense_replace": 3,
    "n_routed_experts": 256, "n_shared_experts": 1, "num_experts_per_tok": 8,
    "q_lora_rank": 1536, "kv_lora_rank": 512, "qk_nope_head_dim": 128, "qk_rope_head_dim": 64, "v_head_dim": 128,
}  # fmt: skip


def deepseek_v3_params(c):
    d, L, h, V = c["hidden_size"], c["num_hidden_layers"], c["num_attention_heads"], c["vocab_size"]
    qr, kvr, vd = c["q_lora_rank"], c["kv_lora_rank"], c["v_head_dim"]
    qk = c["qk_nope_head_dim"] + c["qk_rope_head_dim"]  # chiều Q/K của một đầu: phần thường + phần RoPE
    nd, E, k = c["first_k_dense_replace"], c["n_routed_experts"], c["num_experts_per_tok"]
    expert = 3 * d * c["moe_intermediate_size"]  # một chuyên gia SwiGLU nhỏ
    mla = d * qr + qr * h * qk + d * (kvr + c["qk_rope_head_dim"]) + kvr * h * (c["qk_nope_head_dim"] + vd) + h * vd * d
    parts = {
        "embedding + lớp đầu ra": 2 * V * d,
        "attention (MLA)": L * mla,
        "chuẩn hóa": L * 2 * d + d + L * (qr + kvr),
        "3 FFN dày": nd * 3 * d * c["intermediate_size"],
        "chuyên gia dùng chung": (L - nd) * c["n_shared_experts"] * expert,
        "bộ định tuyến (W_r và độ lệch b)": (L - nd) * (d * E + E),
        "chuyên gia định tuyến": (L - nd) * E * expert,
    }
    total = sum(parts.values())
    return parts, total, total - (L - nd) * (E - k) * expert


parts_ds, total_ds, active_ds = deepseek_v3_params(DSV3)
for name, n in parts_ds.items():
    print(f"{name:>32}: {vn(n, 0):>17} ({pct(n / total_ds, 1)})")
print(f"tổng {vn(total_ds / 1e9, 1)} tỉ · hoạt động {vn(active_ds / 1e9, 1)} tỉ ({pct(active_ds / total_ds, 1)})")
ds = counts["deepseek-v3"]
assert (total_ds, active_ds) == (ds["total"], ds["active"]), "khác với checkpoint: hãy kiểm tra lại config"
assert round(total_ds / 1e9) == 671 and 37e9 <= active_ds < 38e9
mtp = ds["parts"]["mtp"]  # lớp dự đoán nhiều token (MTP), đếm từ header, không dùng khi sinh văn bản thường
print(f"kho Hugging Face còn chứa mô-đun MTP {vn(mtp / 1e9, 2)} tỉ: tổng {vn((total_ds + mtp) / 1e9, 2)} tỉ")
print(f"tệp checkpoint (FP8, kèm hệ số tỉ lệ): {vn(ds['bytes'] / GiB, 1)} GiB")
print(f"lớp ẩn: FFN dày {vn(DSV3['intermediate_size'], 0)}, mỗi chuyên gia {vn(DSV3['moe_intermediate_size'], 0)}")

DeepSeek-V3 dùng **37,6 tỉ trong 671 tỉ** tham số cho mỗi token, khoảng 5,6%: các chuyên gia định tuyến chiếm gần hết
mô hình, còn mỗi token chỉ chạy 8 trong 256 chuyên gia ở mỗi lớp. Con số "685B" trên trang Hugging Face là 671 tỉ cộng
mô-đun MTP (dự đoán nhiều token), mà thẻ mô hình ghi là "14B". Bộ định tuyến có thêm 256 độ lệch $b_i$ mỗi lớp:
đó chính là độ lệch cân bằng tải của mục 6, lưu trong checkpoint dưới tên `e_score_correction_bias`. Dù đã lưu ở FP8
(1 byte mỗi tham số), tệp vẫn nặng 641 GiB, nên DeepSeek-V3 phải chạy trên nhiều máy; báo cáo kỹ thuật giới hạn mỗi
token chỉ được gửi tới tối đa 4 máy, để bớt lưu lượng mạng.

Bảng dưới đọc thẳng từ `moe-counts.json` (không tính lại): 10 checkpoint, mỗi con số được `counts.py` khớp tới từng
tham số giữa công thức config.json và header safetensors.

In [ ]:
print(f"{'mô hình':>16} {'N / k':>8} {'tổng (tỉ)':>10} {'hoạt động':>10} {'tỉ lệ':>7} {'checkpoint':>11}")
for mid, m in counts.items():
    nk = f"{m['E']} / {m['k']}" if m["E"] > 1 else "dày"
    print(
        f"{mid:>16} {nk:>8} {vn(m['total'] / 1e9, 1):>10} {vn(m['active'] / 1e9, 2):>10} "
        f"{pct(m['active'] / m['total'], 1):>7} {vn(m['bytes'] / GiB, 1):>7} GiB"
    )
go, sc = counts["gpt-oss-120b"], counts["llama4-scout"]
print(f"gpt-oss-120b, hoạt động không tính embedding: {vn(go['activeNoEmbed'] / 1e9, 2)} tỉ")
print(f"Llama 4 Scout, phần văn bản + tháp thị giác: {vn((sc['total'] + sc['parts']['vision']) / 1e9, 2)} tỉ")

Hai quy ước cần biết khi đọc thẻ mô hình: gpt-oss gọi 5,13 tỉ là "active" vì không tính ma trận embedding (chỉ là một
phép tra bảng) nhưng tính lớp đầu ra; Llama 4 Scout ghi 109 tỉ vì cộng cả tháp thị giác. Tỉ lệ hoạt động rơi từ 27,6%
(Mixtral, 2 trong 8) xuống 5,6% (DeepSeek-V3, 8 trong 256) và 4,9% (gpt-oss-120b, 4 trong 128): mô hình càng mới
càng dùng nhiều chuyên gia nhỏ và chọn một phần càng nhỏ trong số đó.

## 5 · Huấn luyện trên MNIST: bộ định tuyến sụp

Giờ huấn luyện mạng của mục 3 trên MNIST, đúng công thức của demo: Adam ($\beta_1 = 0{,}9$, $\beta_2 = 0{,}999$),
lô 128 ảnh, $\eta$ cố định, thứ tự ảnh từ `default_rng(1000 + seed)`. Demo huấn luyện 10 epoch (4.290 bước) với 3 seed;
ở đây ta chạy 1 epoch (429 bước), và cứ 43 bước ghi lại **tải** của từng chuyên gia: phần suất
tính mà nó nhận trong 43 bước đó. Một chuyên gia nhận ít hơn ¼ phần công bằng ($\tfrac14 \cdot \tfrac18$ số suất)
được gọi là **chết**, như trong demo.

In [ ]:
def train(seed, k=1, gate="raw", lr=3e-3, alpha=0.0, gamma=0.0, epochs=1, E=8, every=43):
    """Adam như reel/moe/mnist.py. Trả về mạng, độ lệch b (nếu γ > 0) và tải của từng chuyên gia ở bước 1, 43, 86…"""
    P = init_moe(seed, E)
    b1, b2 = 0.9, 0.999
    m = {key: np.zeros_like(w) for key, w in P.items()}
    v = {key: np.zeros_like(w) for key, w in P.items()}
    bias = np.zeros(E, np.float32) if gamma > 0 else None
    order = np.random.default_rng(1000 + seed)
    hist, acc_load, step = [], np.zeros(E), 0
    for _ in range(epochs):
        perm = order.permutation(len(Xtr))
        for b in range(len(Xtr) // BATCH):
            idx = perm[b * BATCH : (b + 1) * BATCH]
            _, _, G, load, _ = loss_grad(P, Xtr[idx], ytr[idx], k, bias, gate, None, alpha)
            step += 1
            lr_t = lr * math.sqrt(1 - b2**step) / (1 - b1**step)
            for key in P:
                m[key] = b1 * m[key] + (1 - b1) * G[key]
                v[key] = b2 * v[key] + (1 - b2) * G[key] ** 2
                P[key] -= (lr_t * m[key] / (np.sqrt(v[key]) + 1e-8)).astype(np.float32)
            if bias is not None:  # DeepSeek-V3 (mục 6): thiếu việc thì tăng b, thừa việc thì giảm
                bias += np.float32(gamma) * np.sign(load.mean() - load).astype(np.float32)
            acc_load += load
            if step % every == 0 or step == 1:
                hist.append(acc_load / acc_load.sum())
                acc_load = np.zeros(E)
    return P, bias, np.array(hist)


def dead(share, E=8):
    return int((share < 0.25 / E).sum())


def stored_hist(name, seed):
    """Tải mỗi 43 bước của một lần chạy 10 epoch trong moe-mnist.json (phần nghìn, uint16)."""
    raw = base64.b64decode(sweep["runs"][name]["seeds"][seed]["hist"])
    return np.frombuffer(raw, np.uint16).reshape(-1, 8) / 1000


R = sweep["runs"]
print(
    "η đã chọn trên tập kiểm định (seed 0):",
    ", ".join(f"{n} {vn(R[n]['lr'], 3)}" for n in ("e8k1_none", "e8k1_aux", "e8k2_bias_sm")),
)
print(
    f"demo: {sweep['setup']['epochs']} epoch = {vn(sweep['setup']['steps'], 0)} bước; tải ghi mỗi {sweep['setup']['every']} bước"
)

### 🤔 Dự đoán trước
Lúc đầu mọi $s_i$ gần bằng $0{,}125$ (mục 3), nên mỗi chuyên gia nhận khoảng 1/8 số ảnh. Huấn luyện top-1 (kiểu
Switch, cổng $g = s_i$), **không có gì ép chia đều**. Sau 43 bước đầu tiên ($43 \cdot 128$ ảnh), bao nhiêu trong 8 chuyên gia
còn nhận được việc?

In [ ]:
live = {}
for seed in range(3):
    live[("e8k1_none", seed)] = train(seed, k=1, gate="raw", lr=R["e8k1_none"]["lr"])
    h_live = live[("e8k1_none", seed)][2]
    h_ref = stored_hist("e8k1_none", seed)[: len(h_live)]
    print(
        f"seed {seed}: tải 43 bước đầu = [{' '.join(vn(v, 3) for v in h_live[1])}] → {dead(h_live[1])} chuyên gia chết"
        f" · cuối epoch 1: {dead(h_live[-1])} chết; lệch lớn nhất so với lần chạy của demo {vn(np.abs(h_live - h_ref).max(), 3)}"
    )
    assert abs(dead(h_live[1]) - dead(h_ref[1])) <= 1, "khác với demo: hãy kiểm tra lại dữ liệu"
dead43 = [dead(live[("e8k1_none", s)][2][1]) for s in range(3)]
print(f"chết sau 43 bước: {dead43} (demo: {[dead(stored_hist('e8k1_none', s)[1]) for s in range(3)]})")
assert min(dead43) >= 5

Chỉ sau 43 bước, **5–7 trong 8 chuyên gia đã chết**: một hai chuyên gia nhận gần hết token. Không có lỗi nào trong
mã. Lúc đầu mọi điểm gần bằng nhau, nhưng một chuyên gia nào đó tình cờ giúp giảm mất mát nhiều hơn một chút cho vài
token. Gradient qua cổng $g_i = s_i$ đẩy $s_i$ của nó lên, nên nó được chọn nhiều hơn. Chỉ chuyên gia được chọn mới
nhận gradient (mục 3), nên nó học nhanh hơn, và lại càng được chọn nhiều hơn. Các chuyên gia còn lại không học gì,
nên không bao giờ hấp dẫn trở lại. Shazeer và cộng sự (2017) đã mô tả đúng vòng này: *"the favored experts are trained
more rapidly and thus are selected even more by the gating network"*.

Hình dưới đọc tải của các lần chạy 10 epoch trong demo (seed 0): mỗi hàng là một chuyên gia, mỗi cột là 43 bước, màu
đậm là phần suất tính lớn. Bên trái là top-1 không cân bằng; bên phải là cùng mạng có thêm mất mát phụ của mục 6.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 2.9), sharey=True)
for ax, name, title in [(axes[0], "e8k1_none", "không cân bằng"), (axes[1], "e8k1_aux", "mất mát phụ α = 0,01")]:
    hs = stored_hist(name, 0)
    im = ax.imshow(hs.T, aspect="auto", cmap=SEQ, vmin=0, vmax=1, interpolation="nearest",
                   extent=(0, len(hs) * 43, 7.5, -0.5))  # fmt: skip
    ax.set(xlabel="bước huấn luyện")
    ax.set_title(f"top-1, {title}\n{dead(hs[-1])} chuyên gia chết ở 43 bước cuối", fontsize=10)
axes[0].set_ylabel("chuyên gia")
axes[0].set_yticks(range(8), [str(e) for e in range(8)])
fig.colorbar(im, ax=axes, fraction=0.03, label="phần suất tính")
vn_axes(fig)
plt.show()
for name in ("e8k1_none", "e8k1_aux"):
    sd = R[name]["seeds"]
    print(
        f"{name}, 10 epoch, 3 seed: chết {[q['dead'] for q in sd]}, "
        f"chuyên gia đông nhất nhận {span([q['maxload'] / 8 for q in sd], 0, 100)}% suất tính trên tập kiểm tra"
    )

Bên trái, hai ba hàng xanh đậm suốt 4.290 bước, còn lại gần như trắng: sụp đổ xảy ra ngay đầu và không bao giờ hồi phục. Ở cuối,
chuyên gia đông nhất nhận 49–100% suất tính (100% ở seed 2: cả mô hình chỉ còn một chuyên gia). Bên phải, mọi hàng đều
xanh nhạt: mỗi chuyên gia nhận khoảng 1/8 số suất.

## 6 · Cân bằng tải

### Mất mát phụ của Switch Transformer

Fedus, Zoph và Shazeer (2021) cộng thêm vào mất mát một số hạng nhỏ nhất khi tải chia đều:

$$\mathcal L_{\text{bal}} = \alpha N \sum_{i=1}^{N} f_i P_i$$

- $f_i$: phần suất tính của lô đã đến chuyên gia $i$. Switch có $k = 1$, nên đó là phần token chọn $i$; với $k = 2$, ta
  đếm cả hai lựa chọn như demo. $f_i$ là một phép đếm: **không khả vi**.
- $P_i$: điểm trung bình $\bar s_i$ mà bộ định tuyến gán cho $i$ trên lô: **khả vi**, nên gradient đi qua đây.
- $\alpha = 0{,}01$ trong Switch. Nhân với $N$ để khi tải chia đều, số hạng luôn bằng $\alpha$, dù có bao nhiêu chuyên gia.

Gradient theo $s_i$ là $\alpha N f_i / n$: chuyên gia càng đông việc thì điểm của nó càng bị kéo xuống mạnh.
Ô dưới tính $\mathcal L_{\text{bal}}$ cho ba lô giả định.

In [ ]:
N_, alpha = 8, 0.01
for name, f, P_ in [
    ("chia đều", np.full(N_, 1 / N_), np.full(N_, 1 / N_)),
    (
        "một nửa vào chuyên gia 0",
        np.r_[0.5, np.full(N_ - 1, 0.5 / (N_ - 1))],
        np.r_[0.5, np.full(N_ - 1, 0.5 / (N_ - 1))],
    ),
    ("tất cả vào chuyên gia 0", np.eye(N_)[0], np.eye(N_)[0]),
]:
    print(f"{name:>25}: L_bal = {vn(alpha * N_ * (f * P_).sum(), 4)}")

Chia đều cho đúng $\alpha$; dồn hết vào một chuyên gia cho $\alpha N$, lớn gấp 8 lần. Huấn luyện lại top-1 với
$\alpha = 0{,}01$ (cùng seed, cùng $\eta$):

In [ ]:
for seed in range(3):
    live[("e8k1_aux", seed)] = train(seed, k=1, gate="raw", lr=R["e8k1_aux"]["lr"], alpha=0.01)
for name in ("e8k1_none", "e8k1_aux"):
    ends = [dead(live[(name, s)][2][-1]) for s in range(3)]
    tops = [live[(name, s)][2][-1].max() * 8 for s in range(3)]
    print(f"{name}, 1 epoch: chết ở 43 bước cuối {ends}; tải lớn nhất {span(tops, 2)} lần phần công bằng")
end_none = [dead(live[("e8k1_none", s)][2][-1]) for s in range(3)]
end_aux = [dead(live[("e8k1_aux", s)][2][-1]) for s in range(3)]
assert max(end_aux) < min(end_none)

Mất mát phụ cứu được các chuyên gia: ở cuối epoch còn 0–2 chuyên gia chết thay vì 5–7, và chuyên gia đông nhất chỉ
nhận khoảng 2 lần phần công bằng (sau 10 epoch của demo: không chuyên gia nào chết). Cái giá là một mục tiêu thứ hai kéo bộ định tuyến khỏi việc chính. Với $\alpha$ quá lớn,
bộ định tuyến chia đều thay vì chia theo nội dung (mục 9 đo điều đó).

### Độ lệch chỉ dùng để chọn của DeepSeek-V3

Wang và cộng sự (2024), rồi DeepSeek-V3, bỏ mất mát phụ và cộng cho mỗi chuyên gia một độ lệch $b_i$ **chỉ dùng lúc
chọn**: $\mathcal T = \operatorname{TopK}(s + b, k)$, còn cổng vẫn tính từ $s$. Sau mỗi bước, theo tải của cả lô:

$$b_i \leftarrow b_i + \gamma \cdot \operatorname{sign}(\bar c - c_i)$$

với $c_i$ là số suất chuyên gia $i$ vừa nhận và $\bar c$ là trung bình: thiếu việc thì $b_i$ tăng, thừa việc thì giảm.
DeepSeek-V3 dùng $\gamma = 0{,}001$ (và $\gamma = 0$ cho 500 tỉ token cuối). $b$ không có trong hàm mất mát, nên không
kéo bộ định tuyến khỏi việc chính. Ô dưới kiểm tra điều đó: nhích $b$ một chút mà không đổi lựa chọn nào, thì mất mát
không đổi tới từng bit.

In [ ]:
Pt = init_moe(seed=3, E=4, H=3, d=4, d_in=5, n_cls=3, dtype=np.float64)
Pt["Wr"] = rng.standard_normal(Pt["Wr"].shape)
Xt, yt_ = rng.standard_normal((7, 5)), rng.integers(0, 3, 7)
b_ = rng.standard_normal(4) * 0.05
base_loss = loss_grad(Pt, Xt, yt_, 2, b_)[0]
for i in range(4):
    b2 = b_.copy()
    b2[i] += 1e-9
    assert loss_grad(Pt, Xt, yt_, 2, b2)[0] == base_loss
print(f"mất mát {vn(base_loss, 6)} giữ nguyên khi nhích từng bᵢ: độ lệch không nhận gradient nào")

Huấn luyện top-2 (cổng chuẩn hóa, như Mixtral) không cân bằng và với độ lệch $\gamma = 0{,}001$ (đây chính là mô hình
mà demo chạy trong trình duyệt, mục 8), 3 seed mỗi cấu hình, rồi đo tải trên 10.000 ảnh kiểm tra:

In [ ]:
for seed in range(3):
    live[("e8k2_none", seed)] = train(seed, k=2, gate="renorm", lr=R["e8k2_none"]["lr"])
    live[("e8k2_bias_sm", seed)] = train(seed, k=2, gate="renorm", lr=R["e8k2_bias_sm"]["lr"], gamma=1e-3)


def test_share(name, seed):
    """Phần suất tính trên tập kiểm tra và độ chính xác của một mô hình vừa huấn luyện."""
    P, bias, _ = live[(name, seed)]
    k = 1 if "k1" in name else 2
    z, C = forward(P, Xte, k, bias, "raw" if k == 1 else "renorm")
    return np.bincount(C["sel"].ravel(), minlength=8) / C["sel"].size, (z.argmax(1) == yte).mean()


share = {key: test_share(*key) for key in live}
for name in ("e8k1_none", "e8k1_aux", "e8k2_none", "e8k2_bias_sm"):
    sh = [share[(name, s)][0] for s in range(3)]
    print(
        f"{name:>13}, 1 epoch: chết {[dead(q) for q in sh]}, tải lớn nhất {span([q.max() * 8 for q in sh], 2)}, "
        f"đúng {span([share[(name, s)][1] for s in range(3)], 2, 100)}%"
    )
assert all(share[("e8k2_bias_sm", s)][0].max() < share[("e8k2_none", s)][0].max() for s in range(3))
print("độ lệch b sau 1 epoch (seed 0):", " ".join(vn(v, 3) for v in live[("e8k2_bias_sm", 0)][1]))

Không cân bằng, top-2 cũng sụp: chỉ còn 3 chuyên gia sống, và chuyên gia đông nhất nhận gần 4 lần phần công bằng (với
$k = 2$, mức tối đa là $N/k = 4$: chuyên gia đó có mặt trong mọi cặp). Độ lệch $b$ giữ cả 8 chuyên gia sống, chỉ bằng
cách đổi lựa chọn của những token đang phân vân, không thêm số hạng nào vào mất mát.

Độ chính xác sau 1 epoch chưa nói được gì (mỗi cấu hình dùng $\eta$ mà demo chọn cho 10 epoch). Các lần chạy
10 epoch của demo:

In [ ]:
def seeds(name, f):
    """Một giá trị cho mỗi seed của một cấu hình trong moe-mnist.json."""
    return [f(q) for q in R[name]["seeds"]]


for name in ("e8k2_none", "e8k2_aux", "e8k2_bias_sm"):
    print(
        f"{name:>13}, 10 epoch: đúng {span(seeds(name, lambda q: q['test']), 2, 100)}%, "
        f"chết {seeds(name, lambda q: q['dead'])}, tải lớn nhất {span(seeds(name, lambda q: q['maxload']), 2)}"
    )

Không giới hạn dung lượng, ba cách nằm trong dao động giữa các seed: sụp đổ không làm giảm độ chính xác ở đây. Mô hình
đã thành một mạng dày nhỏ hơn, vài chuyên gia làm hết việc, và các chuyên gia chết chỉ tốn bộ nhớ. Cái giá hiện ra khi
mỗi chuyên gia chỉ nhận được một số token có hạn.

## 7 · Dung lượng: token bị bỏ khi chuyên gia đầy

Trên phần cứng thật, mỗi chuyên gia xử lý một bộ đệm có kích thước cố định: tensor phải có hình dạng biết trước, và các
chuyên gia thường nằm trên những GPU khác nhau. Switch Transformer cho mỗi chuyên gia

$$C = \Big\lceil \mathrm{cf} \cdot \frac{n\,k}{N} \Big\rceil$$

suất trong một lô $n$ token, với **hệ số dung lượng** cf (capacity factor); cf = 1 là đúng phần công bằng. Token đến
khi chuyên gia đã đầy bị **bỏ** ở lớp đó: chuyên gia không chạy, token đi thẳng qua đường dư sang lớp sau.
`capacity_keep` ở mục 3 xếp mọi lựa chọn thứ nhất của lô vào hàng trước, rồi mới đến lựa chọn thứ hai, như demo.

### 🤔 Dự đoán trước
Giả sử bộ định tuyến cân bằng hoàn hảo *trung bình*: mỗi token chọn 1 trong 8 chuyên gia đều ngẫu nhiên và độc lập.
Lô 128 token, cf = 1, nên $C = 16$, đúng bằng số token trung bình mỗi chuyên gia nhận. Bao nhiêu phần trăm token bị bỏ?

In [ ]:
n_, N_ = 128, 8
sim_rng = np.random.default_rng(0)
for cf in (1.0, 1.25, 1.5, 2.0):
    C_ = math.ceil(cf * n_ / N_)
    # số token một chuyên gia nhận ~ nhị thức(128; 1/8); phần vượt C bị bỏ
    over = sum(math.comb(n_, x) * (1 / N_) ** x * (1 - 1 / N_) ** (n_ - x) * (x - C_) for x in range(C_ + 1, n_ + 1))
    sim = np.mean([(~capacity_keep(sim_rng.integers(0, N_, (n_, 1)), N_, cf)).mean() for _ in range(2000)])
    print(f"cf = {vn(cf, 2)}: C = {C_} → bỏ {pct(over * N_ / n_)} (nhị thức) · mô phỏng 2.000 lô: {pct(sim)}")
    if cf == 1.0:
        drop_uniform = over * N_ / n_
        assert abs(sim - drop_uniform) < 0.005

Khoảng 9%, dù tải *trung bình* bằng đúng dung lượng: do ngẫu nhiên, có chuyên gia nhận 20 token, có chuyên gia chỉ
nhận 12; phần thừa bị bỏ, phần thiếu thì bộ đệm để trống. Vì vậy hệ số dung lượng thường lớn hơn 1 một chút (Switch
thấy cf 1,0–1,25 là tốt nhất), và tải càng lệch thì càng bỏ nhiều. Đo trên các mô hình vừa huấn luyện 1 epoch, với
lô 128 ảnh kiểm tra như demo:

In [ ]:
def eval_capacity(P, k, gate, bias, cf):
    """Độ chính xác và phần suất bị bỏ trên 10.000 ảnh kiểm tra, theo từng lô 128 ảnh."""
    hit = drop = 0
    for b in range(0, len(Xte), BATCH):
        z, C = forward(P, Xte[b : b + BATCH], k, bias, gate, cf)
        hit += int((z.argmax(1) == yte[b : b + BATCH]).sum())
        drop += int((~C["kept"]).sum())
    return hit / len(Xte), drop / (len(Xte) * k)


cap = {}
for name in ("e8k1_none", "e8k1_aux"):
    for seed in range(3):
        P, bias, _ = live[(name, seed)]
        cap[(name, seed)] = eval_capacity(P, 1, "raw", bias, 1.0)
    acc_, drop_ = zip(*(cap[(name, s)] for s in range(3)))
    print(f"{name}, 1 epoch, cf = 1: bỏ {span(drop_, 0, 100)}% suất tính, còn đúng {span(acc_, 1, 100)}%")
assert all(cap[("e8k1_none", s)][1] > cap[("e8k1_aux", s)][1] for s in range(3))

Mô hình đã sụp bỏ phần lớn suất tính: chuyên gia đông nhất nhận gấp mấy lần $C$, phần thừa bị bỏ, còn bộ đệm của các
chuyên gia chết thì để trống. Mô hình có mất mát phụ bỏ ít hơn nhiều. Các lần chạy 10 epoch của demo cho cùng bức
tranh, rõ hơn:

In [ ]:
cfs = sweep["setup"]["cfs"]  # hệ số dung lượng đã đo; None = không giới hạn
i1, i0 = cfs.index(1.0), cfs.index(0.0)
for name in ("e8k1_none", "e8k1_aux", "e8k2_none", "e8k2_bias_sm"):
    print(
        f"{name:>13}, 10 epoch, cf = 1: bỏ {span(seeds(name, lambda q: q['cf'][i1][1]), 0, 100)}%, "
        f"đúng {span(seeds(name, lambda q: q['cf'][i1][0]), 1, 100)}% "
        f"(không giới hạn: {span(seeds(name, lambda q: q['test']), 1, 100)}%; "
        f"cf = 0: {span(seeds(name, lambda q: q['cf'][i0][0]), 1, 100)}%)"
    )
print(f"e8k1_aux: cổng thô sᵢ trung vị {span(seeds('e8k1_aux', lambda q: q['gate1']), 2)}")

Ở cf = 1, top-1 không cân bằng bỏ **64–88%** suất tính và chỉ còn đúng **86,1–96,4%**; có mất mát phụ thì bỏ
**19–20%** mà vẫn đúng **97,1–97,3%**, gần như không mất gì so với không giới hạn. Token bị bỏ không biến mất: nó vẫn
đi tiếp qua đường dư $u = h$.

Nhưng hãy nhìn cột cf = 0, nơi *mọi* suất bị bỏ: mô hình top-1 có mất mát phụ vẫn đúng 97,2–97,3%. Ở mô hình này các
chuyên gia gần như không góp gì (cổng thô $s_i$ của nó có trung vị chỉ 0,30–0,33), nên nó chịu được dung lượng nhỏ
chủ yếu vì không cần đến chuyên gia: thân mạng và đường dư đã đủ cho MNIST. Mô hình top-2 với độ lệch thì cần chúng
(cf = 0: 80,0–89,8%), mà ở cf = 1 vẫn chỉ bỏ 8–10% suất và đúng 97,4–97,5%.

Đường dư quan trọng đến mức nào? Demo còn huấn luyện một kiểu MoE **không có đường dư**, như Jacobs và cộng sự (1991):
mỗi chuyên gia tự đọc 784 điểm ảnh và cho ra 10 logit (784 → 8 → 10), $z = \sum_{i\in\mathcal T} g_i\,
\mathrm{FFN}_i(x)$. Ở đó, token bị bỏ mất luôn phần của chuyên gia đó. Hình dưới vẽ độ chính xác (trái) và phần suất bị
bỏ (phải) theo hệ số dung lượng, top-2, mỗi dấu một seed: đường liền có đường dư, đường gạch không có; xanh chấm tròn
là có cân bằng (mất mát phụ α = 0,01), cam ô vuông là không cân bằng.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.6))
x_cf = [c for c in cfs if c is not None]
series = [("e8k2_aux", BLUE, "o", "-"), ("e8k2_none", ORANGE, "s", "-"), ("h8k2_aux", BLUE, "o", "--"),
          ("h8k2_none", ORANGE, "s", "--")]  # fmt: skip
labels = {"e8k2_aux": "có đường dư, cân bằng", "e8k2_none": "có đường dư, không cân bằng",
          "h8k2_aux": "không đường dư, cân bằng", "h8k2_none": "không đường dư, không cân bằng"}  # fmt: skip
for name, color, marker, ls in series:
    vals = np.array([[q["cf"][i] for i in range(len(x_cf))] for q in R[name]["seeds"]])  # seed × cf × (đúng, bỏ)
    for ax, j in ((a1, 0), (a2, 1)):
        ax.plot(x_cf, 100 * np.median(vals[:, :, j], 0), color=color, ls=ls, lw=1.5, label=labels[name])
        for sd in vals:
            ax.scatter(x_cf, 100 * sd[:, j], color=color, marker=marker, s=18, zorder=3,
                       facecolors="none" if ls == "--" else color)  # fmt: skip
a1.set(xlabel="hệ số dung lượng cf", ylabel="% đúng trên 10.000 ảnh kiểm tra", ylim=(0, 100))
a2.set(xlabel="hệ số dung lượng cf", ylabel="% suất tính bị bỏ", ylim=(0, 100))
a1.set_title("Độ chính xác (top-2)")
a2.set_title("Suất tính bị bỏ (top-2)")
a1.legend(frameon=False, fontsize=8, loc="lower right")
vn_axes(fig)
plt.tight_layout()
plt.show()
for name in ("h8k2_none", "h8k2_aux"):
    print(
        f"{name}: cf = 0 → đúng {span(seeds(name, lambda q: q['cf'][i0][0]), 1, 100)}%; "
        f"cf = 1 → bỏ {span(seeds(name, lambda q: q['cf'][i1][1]), 0, 100)}%, "
        f"đúng {span(seeds(name, lambda q: q['cf'][i1][0]), 1, 100)}%"
    )
print(f"có đường dư, cf = 0 (bỏ hết): đúng {span(seeds('e8k2_aux', lambda q: q['cf'][i0][0]), 1, 100)}%")

Có đường dư, cf = 0 (lớp MoE bị bỏ hẳn) vẫn còn đúng 73–88%, vì thân mạng và lớp đầu ra vẫn chạy. Không có đường dư,
cf = 0 chỉ còn đúng 9,8% (đoán bừa), và ở cf = 1 mô hình không cân bằng tụt xuống 85–87%. Trong Transformer, đường dư
là thứ cho phép bỏ token mà không hỏng mô hình. DeepSeek-V3 đi xa hơn: nhờ độ lệch $b$ cân bằng đủ tốt, nó không bỏ
token nào, cả khi huấn luyện lẫn khi suy luận.

## 8 · Mô hình của demo: int8, chạy trên 10.000 ảnh

Demo chạy trong trình duyệt seed 0 của cấu hình top-2, cổng chuẩn hóa, độ lệch $\gamma = 0{,}001$, huấn luyện 10 epoch.
Để tệp nhỏ, mỗi ma trận được lưu ở **int8** với một hệ số tỉ lệ float16 cho mỗi cột đầu ra (với $A$ và $B$: cho mỗi
chuyên gia), còn vectơ lưu ở float16. Giải nén: $W \approx q \cdot s$, với $q \in \{-127, \dots, 127\}$. Đây cũng là
cách lượng tử hóa mà gpt-oss và DeepSeek-V3 dùng ở mức lớn hơn (MXFP4, FP8) để các chuyên gia vừa bộ nhớ.

In [ ]:
def b64(text, dtype):
    """Giải mã một mảng mà demo lưu dạng base64 (little-endian)."""
    return np.frombuffer(base64.b64decode(text), dtype)


def unpack(t):
    if "f16" in t:
        return b64(t["f16"], np.float16).astype(np.float64).reshape(t["shape"])
    q = b64(t["i8"], np.int8).reshape(t["shape"]).astype(np.float64)
    scale_shape = list(t["shape"])
    scale_shape[-2] = 1  # một hệ số cho mỗi cột đầu ra (và mỗi chuyên gia, với A, B)
    return q * b64(t["scale"], np.float16).astype(np.float64).reshape(scale_shape)


Pd = {key: unpack(t) for key, t in shipped["P"].items() if key != "bias"}
bias_d = unpack(shipped["P"]["bias"])
cfg_d = shipped["cfg"]
n_bytes = sum(len(base64.b64decode(t.get("i8", t.get("f16", "")))) for t in shipped["P"].values())
print(
    f"cấu hình: {cfg_d['E']} chuyên gia {cfg_d['d']} → {cfg_d['H']} → {cfg_d['d']}, top-{cfg_d['k']}, "
    f"{cfg_d['score']}, cổng {cfg_d['gate']}, γ = {vn(cfg_d['gamma'], 3)}"
)
print(f"{vn(sum(w.size for w in Pd.values()), 0)} tham số trong {vn(n_bytes / 1024, 1)} KB (chưa tính hệ số tỉ lệ)")
print("độ lệch b đã học:", " ".join(vn(v, 3) for v in bias_d))

z_d, C_d = forward(Pd, Xte.astype(np.float64), cfg_d["k"], bias_d, cfg_d["gate"])
acc_d = (z_d.argmax(1) == yte).mean()
z0_d, _ = forward(Pd, Xte.astype(np.float64), cfg_d["k"], bias_d, cfg_d["gate"], cf=0.0)
acc0_d = (z0_d.argmax(1) == yte).mean()
print(f"đúng {pct(acc_d)} trên 10.000 ảnh kiểm tra · chỉ đường dư (cf = 0): {pct(acc0_d)}")
assert abs(acc_d - shipped["live"]["test"]) < 0.0005 and abs(acc0_d - shipped["live"]["residual"]) < 0.0005, "khác demo"

x64 = b64(shipped["digits"], np.uint8).reshape(-1, 784)
assert np.array_equal(x64, test_img[:64]) and shipped["labels"] == test_lab[:64].tolist()
print(f"64 ảnh của demo là 64 ảnh kiểm tra đầu tiên; đúng {(z_d[:64].argmax(1) == yte[:64]).sum()}/64")

Mô hình int8 đúng 97,57%, trùng với số demo hiện. Hình dưới là 8 ảnh đầu: dưới mỗi ảnh là hai chuyên gia được chọn và
cổng của chúng. Cổng thứ nhất thường lớn hơn hẳn cổng thứ hai, nhưng hiếm khi bằng 1.

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(10, 1.9))
for t, ax in enumerate(axes):
    ax.imshow(Xte[t].reshape(28, 28), cmap="gray_r", vmin=0, vmax=1)
    (e1, e2), (g1, g2) = C_d["sel"][t], C_d["g"][t]
    ax.set_title(f"số {yte[t]} → {z_d[t].argmax()}\nE{e1}: {vn(g1, 2)}\nE{e2}: {vn(g2, 2)}", fontsize=9)
    ax.axis("off")
plt.show()
print(f"cổng trung vị: thứ nhất {vn(np.median(C_d['g'][:, 0]), 2)}, thứ hai {vn(np.median(C_d['g'][:, 1]), 2)}")
plain = np.sort(np.argsort(-C_d["s"], 1, kind="stable")[:, :2], 1)
print(f"độ lệch b đổi cặp chuyên gia của {pct((plain != np.sort(C_d['sel'], 1)).any(1).mean(), 1)} số ảnh")

Độ lệch $b$ đổi cặp chuyên gia của khoảng một phần năm số ảnh: đó là những token phân vân mà nó đẩy sang chuyên gia
đang thiếu việc. Với phần lớn token, lựa chọn vẫn là của bộ định tuyến.

### Tắt từng chuyên gia

Một chuyên gia quan trọng đến đâu? Tắt nó (mọi suất của nó bị bỏ, như suất tràn dung lượng; token không được chuyển
sang chuyên gia khác) và đo lại:

In [ ]:
acc_off = []
for e in range(8):
    z_off, _ = forward(Pd, Xte.astype(np.float64), 2, bias_d, "renorm", off=np.arange(8) == e)
    acc_off.append((z_off.argmax(1) == yte).mean())
print("tắt chuyên gia 0…7:", " · ".join(pct(a) for a in acc_off))
assert np.allclose(acc_off, shipped["live"]["off"], atol=0.0005), "khác với demo: hãy kiểm tra lại dữ liệu"
print(f"mất nhiều nhất khi tắt chuyên gia {int(np.argmin(acc_off))}: còn {pct(min(acc_off))}")
lost = [acc_d - a for a in acc_off]
print(f"mất {span(lost, 2, 100)} điểm so với {pct(acc_d)} khi đủ 8 chuyên gia")

Không chuyên gia nào là không thể thiếu: tắt bất kỳ chuyên gia nào cũng chỉ mất từ 0,29 tới 2,32 điểm, vì mỗi token
còn chuyên gia thứ hai và đường dư.

## 9 · Chuyên gia học gì?

Tên gọi "chuyên gia" gợi hình ảnh bác sĩ chuyên khoa: mỗi chuyên gia lo một chủ đề. Trên MNIST, "chủ đề" rõ nhất là chữ
số.

### 🤔 Dự đoán trước
Đếm, với mỗi chữ số, chuyên gia nào là lựa chọn thứ nhất của bộ định tuyến. Có 10 chữ số và 8 chuyên gia. Bạn có thấy
"mỗi chuyên gia một chữ số" không? Chữ số đông nhất chiếm khoảng bao nhiêu phần trăm số ảnh của một chuyên gia?

In [ ]:
mat = np.zeros((10, 8), np.int64)
np.add.at(mat, (yte, C_d["sel"][:, 0]), 1)  # hàng: chữ số; cột: chuyên gia thứ nhất
purity = mat.max(0).sum() / mat.sum()  # phần ảnh của mỗi chuyên gia thuộc chữ số đông nhất của nó


def mutual_info(a, b):
    """I(a; b) theo bit, từ bảng đếm chung."""
    joint = np.zeros((a.max() + 1, b.max() + 1))
    np.add.at(joint, (a, b), 1)
    joint /= joint.sum()
    pa, pb = joint.sum(1, keepdims=True), joint.sum(0, keepdims=True)
    nz = joint > 0
    return float((joint[nz] * np.log2(joint[nz] / (pa @ pb)[nz])).sum())


I_d = mutual_info(yte, C_d["sel"][:, 0])
rows_ = mat / np.linalg.norm(mat, axis=1, keepdims=True)
cos = rows_ @ rows_.T  # hai chữ số giống nhau khi chúng được gửi tới cùng những chuyên gia
pairs = sorted(((cos[a, b], a, b) for a in range(10) for b in range(a + 1, 10)), reverse=True)[:4]
print(f"độ thuần: {pct(purity, 1)} · I(chữ số; chuyên gia) = {vn(I_d, 2)} bit (log₂10 = {vn(math.log2(10), 2)} bit)")
print("4 cặp chữ số có hàng định tuyến giống nhau nhất:", ", ".join(f"{a}–{b} ({vn(c, 2)})" for c, a, b in pairs))
ref_mat = np.array(R["e8k2_bias_sm"]["seeds"][0]["mat"])  # cùng mô hình ở float32, trước khi nén int8
print(f"khác ma trận float32 của demo ở {np.abs(mat - ref_mat).sum()} trên 10.000 ảnh")
assert abs(I_d - R["e8k2_bias_sm"]["seeds"][0]["I"]) < 0.01 and (pairs[0][1], pairs[0][2]) == (3, 5)

Không có "mỗi chuyên gia một chữ số". Hình dưới vẽ ma trận đó, mỗi hàng chia cho số ảnh của chữ số ấy (hàng cộng lại
100%; ô ghi số khi từ 10% trở lên). Vài chữ số gần như luôn đi tới một chuyên gia (số 0 tới E7, số 1 tới E4, số 9 tới
E5), nhưng chính các chuyên gia đó cũng nhận chữ số khác, và vài chữ số (2, 5, 8) chia ra nhiều chuyên gia.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.6))
frac = mat / mat.sum(1, keepdims=True)
im = ax.imshow(100 * frac, cmap=SEQ, vmin=0, vmax=100)
for (r, c), v in np.ndenumerate(frac):
    if v >= 0.1:
        ax.text(c, r, f"{100 * v:.0f}", ha="center", va="center", fontsize=8, color="white" if v > 0.5 else "black")
ax.set_xticks(range(8), [f"E{e}" for e in range(8)])
ax.set_yticks(range(10), [str(d) for d in range(10)])
ax.set(xlabel="chuyên gia được chọn đầu tiên", ylabel="chữ số")
ax.set_title("Mô hình của demo: chữ số → chuyên gia (% mỗi hàng)", fontsize=11)
fig.colorbar(im, ax=ax, fraction=0.04, label="% ảnh của chữ số")
vn_axes(fig)
plt.show()

Hàng 3 và hàng 5 giống nhau nhất (cùng dồn vào E0 và E1), và E5 nhận gần hết số 9 cùng một nửa số 4: bộ định tuyến gom
những chữ số **trông giống nhau**, vì nó chỉ nhìn đặc trưng $h$ của thân mạng, không biết nhãn. Định tuyến vẫn mang
thông tin về chữ số (1,64 bit, gần một nửa của log₂10), nhưng đó là thông tin về hình dạng nét.

Một mô hình có thể là tình cờ. Demo đếm, trên 4 cấu hình top-2 có cân bằng × 3 seed, 4 cặp chữ số có hàng định tuyến
giống nhau nhất của mỗi lần chạy:

In [ ]:
PAIR_RUNS = ["e8k2_bias_sm", "e8k2_aux", "e8k2_aux01", "e8k2_bias"]
pair_count, n_runs = {}, 0
for name in PAIR_RUNS:
    for q in R[name]["seeds"]:
        n_runs += 1
        for a, b, _ in q["pairs"]:
            pair_count[(a, b)] = pair_count.get((a, b), 0) + 1
top = sorted(pair_count.items(), key=lambda kv: -kv[1])[:3]
print(f"{n_runs} lần chạy:", ", ".join(f"{a}–{b} ở {n}/{n_runs}" for (a, b), n in top))
for name, label in [("e8k2_frozen", "bộ định tuyến ngẫu nhiên, không học"), ("e8k2_bias_sm", "độ lệch γ = 0,001"),
                    ("e8k2_aux01", "mất mát phụ α = 0,1"), ("e8k2_aux1", "mất mát phụ α = 1")]:  # fmt: skip
    I_s = seeds(name, lambda q: q["I"])
    print(
        f"{label:>36}: I(chữ số; chuyên gia) = {span(I_s, 2)} bit, đúng {span(seeds(name, lambda q: q['test']), 2, 100)}%"
    )

3 với 5 đi cùng nhau ở 11/12 lần chạy, 4 với 9 ở 9/12: đúng những cặp mà Bài 01 thấy khó tách bằng một nơ-ron. Hai
so sánh nữa:
- Bộ định tuyến **không học** (ma trận ngẫu nhiên cố định) vẫn mang 0,44–0,66 bit về chữ số, vì thân mạng phía trước
  vẫn học, nên các chữ số khác nhau rơi vào vùng khác nhau của cả một phép chiếu ngẫu nhiên. Học thêm khoảng 1 bit.
- Mất mát phụ **quá mạnh** ($\alpha = 1$) kéo xuống còn 0,68–0,79 bit, so với 1,52–1,63 bit ở $\alpha = 0{,}1$: bộ
  định tuyến bị ép chia đều thay vì chia theo nội dung. Đó là lý do DeepSeek-V3 chuyển sang độ lệch $b$, và giữ một
  mất mát phụ rất nhỏ ($\alpha = 0{,}0001$) chỉ để tránh lệch quá nặng trong từng chuỗi.

Mô hình ngôn ngữ thật cũng vậy. Mixtral đo trên các miền của The Pile (ArXiv, PubMed, PhilPapers…) và *"không thấy mẫu
rõ nào trong việc giao chuyên gia theo chủ đề"*. Việc định tuyến bám theo cú pháp: `self` trong Python, các token thụt
lề trong mã thường đi cùng một chuyên gia, và hai token liền nhau hay chọn cùng chuyên gia. OLMoE (64 chuyên gia nhỏ,
huấn luyện từ đầu) thì thấy chuyên môn theo miền rõ hơn ở những miền đặc biệt như arXiv và GitHub, gần như đều ở miền
chung như C4, và chuyên môn mạnh theo từng loại token. "Chuyên gia" là một tên gọi, không phải một bác sĩ chuyên khoa.

## 10 · MoE có hơn mạng dày không?

Trên MNIST, câu trả lời phải đo, không đoán. Demo so các cấu hình MoE với mạng dày cùng thân mạng, một FFN có lớp ẩn
$H$ thay cho 8 chuyên gia: $H = 64$ có cùng **số tham số hoạt động** với MoE top-2 (hai chuyên gia 32), $H = 256$ có
cùng **tổng tham số** (tám chuyên gia 32). Hình dưới vẽ độ chính xác trên tập kiểm tra, mỗi dấu một seed: mạng dày là
hình thoi xanh lá, MoE là chấm tròn xanh; nhãn ghi số tham số chạy cho mỗi ảnh và tổng.

In [ ]:
CMP = ["dense32", "dense64", "dense128", "dense256", "e8k2_aux", "e8k2_bias_sm", "e16k4_aux", "e32k8_aux",
       "e64k16_aux", "s1e15k3_aux", "e2k2_aux", "e4k2_aux", "e16k2_aux", "e32k2_aux"]  # fmt: skip


def label(name):
    c, p = R[name]["cfg"], R[name]["params"]
    if c["E"] == 1:
        return f"dày H {c['H']} · {vn(p['total'], 0)}"
    kind = f"{c['E']} × H {c['H']} top-{c['k']}" + (" + 1 dùng chung" if c["shared"] else "")
    kind += " · độ lệch" if c["bal"] == "bias" else ""
    return f"{kind} · {vn(p['active'], 0)} / {vn(p['total'], 0)}"


fig, ax = plt.subplots(figsize=(9, 5.2))
for y, name in enumerate(CMP):
    is_dense = R[name]["cfg"]["E"] == 1
    acc_s = seeds(name, lambda q: 100 * q["test"])
    ax.scatter(acc_s, [y] * 3, marker="D" if is_dense else "o", color=GREEN if is_dense else BLUE, s=36, zorder=3)
    ax.axhline(y, color=GREY, lw=0.4, alpha=0.4, zorder=1)
ax.set_yticks(range(len(CMP)), [label(n) for n in CMP], fontsize=8.5)
ax.invert_yaxis()
ax.set(xlabel="% đúng trên 10.000 ảnh kiểm tra (3 seed)")
ax.set_title("Có đường dư: MoE và mạng dày (nhãn: tham số chạy / tổng)", fontsize=11)
vn_axes(fig)
plt.tight_layout()
plt.show()
for name in ("dense64", "dense128", "dense256", "e8k2_aux", "e8k2_bias_sm"):
    print(f"{label(name):>40}: {span(seeds(name, lambda q: q['test']), 2, 100)}%")
moe_top2 = seeds("e8k2_aux", lambda q: q["test"]) + seeds("e8k2_bias_sm", lambda q: q["test"])
print(f"MoE top-2 (mất mát phụ và độ lệch, 6 lần chạy): {span(moe_top2, 2, 100)}%")

Có đường dư, MoE top-2 đúng **97,40–97,57%**, hơn mạng dày cùng số tham số hoạt động ($H = 64$: 96,81–97,33%) và cả
mạng dày cùng tổng tham số ($H = 256$: 97,15–97,38%), nhưng chỉ hơn vài phần mười điểm, vừa ra ngoài dao động giữa các
seed. Mạng dày $H = 128$ (97,07–97,66%) chồng lên khoảng của MoE. Chia nhỏ chuyên gia hay thêm chuyên gia dùng chung
cũng không tạo khác biệt rõ ở cỡ này. Còn khi **không có đường dư**:

In [ ]:
for name, what in [("hd24", "mạng đơn 784 → 24 → 10 (cùng phần chạy)"), ("h8k2_aux", "MoE 8 chuyên gia 784 → 8 → 10, top-2"),
                   ("hd64", "mạng đơn 784 → 64 → 10")]:  # fmt: skip
    p = R[name]["params"]
    print(
        f"{what:>40}: chạy {vn(p['active'], 0):>6}, tổng {vn(p['total'], 0):>6} tham số → "
        f"{span(seeds(name, lambda q: q['test']), 2, 100)}%"
    )


def choices(N, k):
    n = math.comb(N, k)
    return vn(n, 0) if n < 1e6 else sci(n)


print(f"8 × H 32 top-2: {choices(8, 2)} cách chọn")
for name in ("e16k4_aux", "e32k8_aux", "e64k16_aux", "s1e15k3_aux", "e2k2_aux", "e4k2_aux", "e16k2_aux", "e32k2_aux"):
    c = R[name]["cfg"]
    print(
        f"{label(name):>40}: {span(seeds(name, lambda q: q['test']), 2, 100)}% · "
        f"{choices(c['E'], c['k'])} cách chọn · chết {seeds(name, lambda q: q['dead'])}"
    )

Không có đường dư, MoE thắng mạng đơn cùng phần chạy, nhưng **thua một mạng đơn ít tổng tham số hơn** (784 → 64 → 10:
50.890 tham số so với 57.232). Một cách giải thích: mỗi chuyên gia chỉ thấy khoảng $k/N$ = một phần tư số ảnh. Từ 2 đến
32 chuyên gia (top-2), độ chính xác nằm trong dao động giữa các seed, và với 32 chuyên gia lại có chuyên gia chết dù
có mất mát phụ.

Chia nhỏ chuyên gia (DeepSeekMoE) cho nhiều cách kết hợp hơn: chọn 2 trong 8 có 28 tổ hợp, 4 trong 16 có 1.820, 16 trong
64 có khoảng $4{,}9 \cdot 10^{14}$, nhưng ở cỡ MNIST điều đó không đổi độ chính xác. Lợi thế lớn của MoE mà các bài báo báo cáo đến từ dữ liệu rất lớn: Switch Transformer
huấn luyện nhanh tới 7 lần với cùng phép tính, trên hơn 180 tỉ token. MNIST quá nhỏ để thấy điều đó; cái nó cho thấy
rõ là cơ chế: sụp đổ, cân bằng, dung lượng, và những gì chuyên gia thật sự học.

## 11 · Thử thay đổi

Các ô dưới chạy lại `train()` (1 epoch, seed 0) với tham số khác và đo trên 10.000 ảnh kiểm tra. Bạn có thể sửa giá trị
rồi chạy lại. Một seed và một epoch thì chỉ cho thấy xu hướng: hãy thêm seed trước khi kết luận.

In [ ]:
def summary(P, bias, k, gate):
    """Độ chính xác, chuyên gia chết, tải lớn nhất và I(chữ số; chuyên gia) trên 10.000 ảnh kiểm tra."""
    z, C = forward(P, Xte, k, bias, gate)
    sh = np.bincount(C["sel"].ravel(), minlength=8) / C["sel"].size
    acc = (z.argmax(1) == yte).mean()
    return f"đúng {pct(acc)}, chết {dead(sh)}, tải lớn nhất {vn(sh.max() * 8, 2)}, I = {vn(mutual_info(yte, C['sel'][:, 0]), 2)} bit"


def trial(k=2, gate="renorm", lr=1e-3, alpha=0.0, gamma=0.0, seed=0):
    P, bias, _ = train(seed, k=k, gate=gate, lr=lr, alpha=alpha, gamma=gamma)
    return summary(P, bias, k, gate)


print("Mất mát phụ α (top-2):")
for a in (0.01, 0.1, 1.0):
    print(f"  α = {vn(a, 2)}: {trial(alpha=a)}")
print("Tốc độ cập nhật độ lệch γ (top-2):")
for gm in (1e-4, 1e-3, 1e-2):
    print(f"  γ = {vn(gm, 4)}: {trial(gamma=gm)}")

$\alpha$ lớn hơn thì tải đều hơn, nhưng ở $\alpha = 1$ định tuyến mang ít thông tin về chữ số hơn ở $\alpha = 0{,}1$,
giống kết quả 10 epoch × 3 seed của demo ở mục 9. Với $\gamma = 0{,}0001$, độ lệch quá chậm: sau 429 bước mỗi $b_i$ chỉ
đi được nhiều nhất $0{,}0429$, trong khi điểm của chuyên gia thắng đã hơn hẳn, nên mô hình kịp sụp trước.

### Top-1 với cổng chuẩn hóa

Vì sao Switch (top-1) giữ cổng thô $g = s_i$? Thử chuẩn hóa: với $k = 1$, $g = s_i / s_i = 1$ cho mọi token.

In [ ]:
P_r, _, _ = train(0, k=1, gate="renorm", lr=3e-3)
moved = np.abs(P_r["Wr"] - init_moe(0)["Wr"]).max()
print(f"top-1, cổng chuẩn hóa: W_r thay đổi nhiều nhất {vn(moved, 6)} sau 429 bước · {summary(P_r, None, 1, 'renorm')}")
P_raw, _, _ = live[("e8k1_none", 0)]  # cùng seed, cổng thô, từ mục 5
print(f"top-1, cổng thô (mục 5):            · {summary(P_raw, None, 1, 'raw')}")
assert moved == 0.0

$W_r$ không đổi một bit nào. Cổng luôn bằng 1 nên $\partial g/\partial s = 0$: bộ định tuyến không nhận gradient nào từ
bài toán, và nó chỉ còn là một phép chiếu ngẫu nhiên cố định của $h$, như bộ định tuyến không học ở mục 9. Vì không
có vòng tự củng cố qua $W_r$, nó không sụp nặng như cổng thô (dòng thứ hai: 6 chuyên gia chết), nhưng nó cũng không
học chia việc theo nội dung.
Gradient đến bộ định tuyến *chỉ* đi qua giá trị của cổng; Shazeer và cộng sự từng nghĩ cần $k \ge 2$ để so sánh hai
chuyên gia, còn Switch cho thấy $k = 1$ đủ, miễn là đầu ra được nhân với $s_i$.

## 12 · Giới hạn và bước tiếp theo

MoE làm lớp FFN rẻ đi trên mỗi token, nhưng không đụng tới attention. Mỗi token mới vẫn phải đọc lại khóa và giá trị
của mọi token trước nó (Bài 17), nên khi ngữ cảnh dài ra, chính attention và bộ nhớ đệm KV của nó thành phần đắt nhất.
Bài 25 hỏi: có thay attention bằng một trạng thái cỡ cố định, như RNN, mà vẫn nhớ đủ không?

MoE cũng không giảm bộ nhớ: mọi chuyên gia phải được nạp sẵn (87,0 GiB cho Mixtral, 641 GiB cho DeepSeek-V3), và đưa
token qua lại giữa các GPU giữ chuyên gia là một chi phí riêng mà notebook này không đo.

## Tóm tắt
- Lớp MoE: bộ định tuyến tuyến tính $s = \operatorname{softmax}(hW_r)$, chọn top-k, cổng chuẩn hóa, $u = h + \sum g_i
  \mathrm{FFN}_i(h)$. Chuyên gia không được chọn không chạy và không nhận gradient.
- Mixtral 8x7B: 46,7 tỉ tham số (không phải 56), 12,9 tỉ mỗi token; top-2 tốn 1,78 lần top-1; trọng số 87,0 GiB.
  DeepSeek-V3: 671 / 37,6 tỉ. Tính từ config.json, khớp tới từng tham số với checkpoint.
- Không cân bằng, bộ định tuyến sụp: 5–7 trong 8 chuyên gia chết sau 43 bước, và không hồi phục.
- Mất mát phụ $\alpha N \sum f_i P_i$ (Switch) hoặc độ lệch chỉ dùng để chọn (DeepSeek-V3) giữ mọi chuyên gia sống;
  $\alpha$ quá lớn làm định tuyến mất thông tin.
- Dung lượng $C = \lceil \mathrm{cf} \cdot nk/N \rceil$: ở cf = 1, mô hình đã sụp bỏ 64–88% suất tính (86,1–96,4% đúng),
  mô hình cân bằng bỏ 19–20% (97,1–97,3%): token bị bỏ đi qua đường dư, và ở MNIST các chuyên gia của nó góp rất
  ít. Định tuyến đều ngẫu nhiên vẫn bỏ 9%.
- Chuyên gia gom chữ số trông giống nhau (3–5 ở 11/12 lần chạy, 4–9 ở 9/12), không học "chủ đề".
- Trên MNIST, MoE chỉ hơn mạng dày vài phần mười điểm khi có đường dư, và thua khi không có.

## Câu hỏi ôn tập
1. Vì sao Mixtral 8x7B có 46,7 tỉ tham số chứ không phải 8 × 7 = 56 tỉ?
2. Vì sao top-2 tốn 1,78 lần top-1 chứ không phải 2 lần?
3. Vì sao một bộ định tuyến không được cân bằng lại tự sụp, và sụp rất sớm?
4. Độ lệch $b_i$ của DeepSeek-V3 cân bằng tải bằng cách nào, mà không kéo mô hình khỏi mục tiêu chính?
5. Tải trung bình đúng bằng dung lượng (cf = 1), vậy sao vẫn có token bị bỏ? Token bị bỏ đi đâu?
6. Top-1 với cổng chuẩn hóa thì bộ định tuyến học được gì?

<details><summary>Gợi ý đáp án</summary>

1. Chỉ các lớp FFN được nhân 8 lần; attention, embedding và chuẩn hóa (khoảng 1,6 tỉ) dùng chung.
2. Phần dùng chung chạy một lần dù $k$ bằng mấy; chỉ phần chuyên gia tăng gấp đôi.
3. Chuyên gia được chọn nhận gradient và học nhanh hơn, nên cổng của nó tăng và nó càng được chọn; chuyên gia không được
   chọn không học gì nên không bao giờ hấp dẫn trở lại. Lúc đầu mọi điểm gần bằng nhau, nên một chênh lệch nhỏ đủ châm ngòi.
4. $b_i$ chỉ cộng vào điểm lúc chọn top-k, không vào cổng, nên không có trong mất mát và không nhận gradient. Sau mỗi
   bước, $b_i$ tăng $\gamma$ nếu chuyên gia thiếu việc, giảm $\gamma$ nếu thừa.
5. Do ngẫu nhiên, có chuyên gia nhận nhiều hơn $C$ trong một lô, phần thừa bị bỏ, còn chuyên gia khác để trống chỗ. Token
   bị bỏ không qua chuyên gia đó ở lớp này; nó đi tiếp qua đường dư (Switch).
6. Không gì cả: cổng luôn bằng 1 nên $W_r$ không nhận gradient nào; lựa chọn giữ như lúc khởi tạo.
</details>

## Tìm hiểu thêm
- R. Jacobs, M. Jordan, S. Nowlan & G. Hinton (1991), [Adaptive Mixtures of Local Experts](https://www.cs.toronto.edu/~hinton/absps/jjnh91.pdf).
- N. Shazeer và cộng sự (2017), [Outrageously Large Neural Networks: The Sparsely-Gated Mixture-of-Experts Layer](https://arxiv.org/abs/1701.06538):
  top-k có nhiễu và vòng tự củng cố của bộ định tuyến.
- W. Fedus, B. Zoph & N. Shazeer (2021), [Switch Transformers](https://arxiv.org/abs/2101.03961): top-1, dung lượng,
  mất mát phụ $\alpha N \sum f_i P_i$.
- A. Jiang và cộng sự (2024), [Mixtral of Experts](https://arxiv.org/abs/2401.04088): phân tích định tuyến theo miền và cú pháp.
- D. Dai và cộng sự (2024), [DeepSeekMoE](https://arxiv.org/abs/2401.06066): chuyên gia nhỏ và chuyên gia dùng chung.
- L. Wang và cộng sự (2024), [Auxiliary-Loss-Free Load Balancing](https://arxiv.org/abs/2408.15664) và
  DeepSeek-AI (2024), [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437).
- N. Muennighoff và cộng sự (2024), [OLMoE](https://arxiv.org/abs/2409.02060): chuyên môn của chuyên gia, đo kỹ.
- Hugging Face (2023), [Mixture of Experts Explained](https://huggingface.co/blog/moe).

**Bài tiếp theo:** [Bài 25 · RNN trở lại: linear attention & Mamba](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rnn-tro-lai/):
thay attention bằng một trạng thái cỡ cố định.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")